## Week 2 Lab 2 - Mail merge campaign

Extendí el Sales Manager del lab para que envíe un email personalizado a cada contacto de una lista (mail merge).

- **Herramientas nuevas:** `get_contacts`, `preview_merge` y `mail_merge_and_send`.
- **Agentes nuevos (agents-as-tools):** `template_writer` y `html_converter`.
- **Seguridad:** los contactos son ficticios y sus emails son alias de mi propia dirección (`EMAIL_ADDRESS` del `.env`), así que ningún correo llega a personas reales.

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from email.message import EmailMessage
import csv
import os
import requests
import smtplib

load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"

In [ ]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")
USE_EMAIL = bool(EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD)

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print("Sending by email" if USE_EMAIL else "Email not configured: sending push notifications instead")


def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)


def send_email_to(to, subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = to
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")
    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)


def send_message_to(to, subject, text_body, html_body):
    if USE_EMAIL:
        send_email_to(to, subject, text_body, html_body)
    else:
        push(f"To: {to}\nSubject: {subject}\n\n{text_body}")

### Lista de contactos

Los emails se generan como alias de mi propia dirección (por ejemplo `usuario+ana@gmail.com`), así todos llegan a mi bandeja.

In [ ]:
PROSPECTS = [
    ("Ana Gómez", "ana", "Acme Fintech", "CTO"),
    ("Luis Pérez", "luis", "DataCorp", "Head of Security"),
    ("Marta Ruiz", "marta", "HealthNow", "Compliance Manager"),
]


def test_alias(alias):
    user, domain = (EMAIL_ADDRESS or "demo@example.com").split("@")
    return f"{user}+{alias}@{domain}"


with open("contacts.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "email", "company", "role"])
    for name, alias, company, role in PROSPECTS:
        writer.writerow([name, test_alias(alias), company, role])


def load_contacts():
    with open("contacts.csv", encoding="utf-8") as f:
        return list(csv.DictReader(f))


load_contacts()

### Los 3 redactores del lab, como herramientas

In [ ]:
intro = """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

sales_agent1 = Agent(name="Professional Sales Agent", instructions=intro + "Your email style is professional, serious, with gravitas and credibility.", model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=intro + "Your email style is witty, engaging, and humorous.", model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=intro + "Your email style is concise, to the point, in the style of a busy senior executive.", model=MODEL_NAME)

description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

### Herramientas de mail merge

Uso `replace` en lugar de `.format()` porque el HTML puede contener llaves de CSS que romperían `.format()`.

In [ ]:
MERGE_FIELDS = ("name", "company", "role")
MAX_RECIPIENTS = 20


def fill(template, contact):
    for field in MERGE_FIELDS:
        template = template.replace("{" + field + "}", contact[field])
    return template


@function_tool
def get_contacts() -> str:
    """Return the prospects in the mailing list, with the fields available for the mail merge"""
    contacts = load_contacts()
    return f"{len(contacts)} contacts. Fields: {', '.join(MERGE_FIELDS)}.\n" + "\n".join(
        f"- {c['name']} ({c['role']} at {c['company']})" for c in contacts
    )


@function_tool
def preview_merge(subject: str, text_template: str) -> str:
    """Show how the email looks for the first contact, to check that the placeholders are replaced correctly

    Args:
        subject: The subject, may contain {name}, {company}, {role}
        text_template: The plain text body with placeholders
    """
    contact = load_contacts()[0]
    return f"Subject: {fill(subject, contact)}\n\n{fill(text_template, contact)}"


@function_tool
def mail_merge_and_send(subject: str, text_template: str, html_template: str) -> str:
    """Send a personalized email to every contact in the list, replacing {name}, {company} and {role}

    Args:
        subject: The subject, may contain placeholders
        text_template: The plain text body with placeholders
        html_template: The HTML body with placeholders
    """
    contacts = load_contacts()[:MAX_RECIPIENTS]
    for contact in contacts:
        send_message_to(contact["email"], fill(subject, contact), fill(text_template, contact), fill(html_template, contact))
    return f"Sent {len(contacts)} personalized emails"

### Agentes nuevos, usados como herramientas

In [ ]:
template_agent = Agent(
    name="Template Writer",
    instructions="""You turn a cold sales email into a mail-merge template.
Replace the recipient's name with {name}, their company with {company} and their job title with {role}.
Use each placeholder at least once where it sounds natural. Keep everything else unchanged.
Reply with the subject line on the first line, then the body.""",
    model=MODEL_NAME,
)

html_agent = Agent(
    name="HTML Converter",
    instructions="""You convert a plain text email into a clean, simple HTML email body.
Keep the placeholders {name}, {company} and {role} exactly as they are. Reply with the HTML only.""",
    model=MODEL_NAME,
)

template_tool = template_agent.as_tool(tool_name="template_writer", tool_description="Turn an email into a mail-merge template with placeholders")
html_tool = html_agent.as_tool(tool_name="html_converter", tool_description="Convert a plain text email template to HTML")

### El Campaign Manager

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. You run a personalized email campaign to a list of prospects.
"""

task = """
Follow these steps:

1. Use get_contacts to see who is on the list and which merge fields are available.
2. Use each of the three sales_email_writer tools to generate a draft. Wait until all three are ready.
3. Choose the single best draft.
4. Use template_writer to turn it into a template with {name}, {company} and {role}.
5. Use html_converter to create the HTML version of the template.
6. Use preview_merge to check the result for the first contact. If a placeholder was not replaced, fix the template.
7. Use mail_merge_and_send exactly once to send the campaign.
"""

tools = [tool1, tool2, tool3, template_tool, html_tool, get_contacts, preview_merge, mail_merge_and_send]
campaign_manager = Agent(name="Campaign Manager", instructions=instructions, tools=tools, model=MODEL_NAME)

In [ ]:
with trace("Mail merge campaign"):
    result = await Runner.run(campaign_manager, task)
print(result.final_output)